# GPU-Oriented Full Dataset Preprocessing

This notebook extracts the full multilabel dataset into
`D:/hvc/datasets/data-from-juniors/full_dataset_frames` and writes the same
manifest files expected by `new_model.ipynb`.

It tries a real GPU path first:

- GPU decode via `decord` if available
- GPU resize via PyTorch CUDA tensors

If the notebook environment does not have a working GPU video backend, it falls
back to the fast sequential CPU extractor instead of the slow random-seek loop.

The notebook is resumable:

- existing extracted frame folders are reused
- manifests are checkpointed every 100 videos


In [ ]:
from pathlib import Path
import random
import shutil
import warnings

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning)


DATA_ROOT = Path(r"D:\hvc\datasets\data-from-juniors")
VIDEO_ROOT = DATA_ROOT / "videos"
LABEL_XLSX = DATA_ROOT / "labels_final.xlsx"
FRAMES_ROOT = DATA_ROOT / "full_dataset_frames"

BASE_MANIFEST_PATH = DATA_ROOT / "full_dataset_base_manifest.csv"
SKIPPED_REPORT_PATH = DATA_ROOT / "full_dataset_skipped_videos.csv"

CFG = {
    "seed": 42,
    "extract_frames_per_video": 16,
    "extract_size": 256,
    "jpeg_quality": 95,
    "overwrite_existing_frames": False,
    "checkpoint_every": 100,
}

VALID_VIDEO_SUFFIXES = {".mp4", ".webm", ".avi", ".mov", ".mkv"}

random.seed(CFG["seed"])
np.random.seed(CFG["seed"])
torch.manual_seed(CFG["seed"])

FRAMES_ROOT.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("data root:", DATA_ROOT)
print("video root:", VIDEO_ROOT)
print("label file:", LABEL_XLSX)
print("frames root:", FRAMES_ROOT)
print("device:", DEVICE)


data root: D:\hvc\datasets\data-from-juniors
video root: D:\hvc\datasets\data-from-juniors\videos
label file: D:\hvc\datasets\data-from-juniors\labels_final.xlsx
frames root: D:\hvc\datasets\data-from-juniors\full_dataset_frames
device: cuda


In [ ]:
def normalize_video_id(value):
    text = str(value).strip()
    if text.endswith(".0"):
        integer_candidate = text[:-2]
        if integer_candidate.isdigit():
            text = integer_candidate
    return text


def build_video_lookup(video_root):
    extension_priority = {".mp4": 0, ".webm": 1, ".avi": 2, ".mov": 3, ".mkv": 4}
    best_files = {}

    for path in video_root.iterdir():
        if not path.is_file():
            continue
        suffix = path.suffix.lower()
        if suffix not in VALID_VIDEO_SUFFIXES:
            continue

        key = path.stem
        rank = extension_priority.get(suffix, 99)
        current = best_files.get(key)
        if current is None or rank < current[0]:
            best_files[key] = (rank, path)

    return {key: value[1] for key, value in best_files.items()}


labels_df = pd.read_excel(LABEL_XLSX, sheet_name=0, engine="openpyxl")
labels_df["source_video_id"] = labels_df["video_id"].map(normalize_video_id)

label_columns = [column for column in labels_df.columns if column not in {"video_id", "source_video_id"}]
labels_df[label_columns] = labels_df[label_columns].fillna(0).astype(int)
labels_df["label_count"] = labels_df[label_columns].sum(axis=1)
labels_df = labels_df[labels_df["label_count"] > 0].copy()

video_lookup = build_video_lookup(VIDEO_ROOT)
labels_df["video_path"] = labels_df["source_video_id"].map(lambda value: video_lookup.get(value))
labels_df["has_video_file"] = labels_df["video_path"].notna()

print(f"rows with at least one label: {len(labels_df):,}")
print(f"rows with matching video file: {int(labels_df['has_video_file'].sum()):,}")
print(f"rows missing a video file: {int((~labels_df['has_video_file']).sum()):,}")

display(
    pd.DataFrame(
        {
            "positives": labels_df[label_columns].sum().astype(int),
            "prevalence_%": (labels_df[label_columns].mean() * 100).round(2),
        }
    ).sort_values("positives", ascending=False)
)

labels_df.head()


rows with at least one label: 5,139
rows with matching video file: 5,118
rows missing a video file: 21


,positives,prevalence_%
humour,4415,85.91
sensitive,702,13.66
anger,373,7.26
derogatory__lang,351,6.83
generic,318,6.19
positive,190,3.70
emotional,165,3.21
threat,150,2.92
political_hate,131,2.55
gender_hate,109,2.12


,video_id,generic,humour,positive,sensitive,derogatory__lang,threat,sexuality_hate,nationality_hate,caste_based_hate,...,anger,emotional,social_hate,controversial,indv_hate,gender_hate,source_video_id,label_count,video_path,has_video_file
0,1,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,1,D:\hvc\datasets\data-from-juniors\videos\1.mp4,True
1,2,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,2,1,D:\hvc\datasets\data-from-juniors\videos\2.mp4,True
2,3,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,3,1,D:\hvc\datasets\data-from-juniors\videos\3.mp4,True
3,4,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,4,1,D:\hvc\datasets\data-from-juniors\videos\4.mp4,True
4,5,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,5,1,D:\hvc\datasets\data-from-juniors\videos\5.mp4,True


In [ ]:
def uniform_indices(frame_count, num_frames):
    if frame_count <= 1:
        return [0] * num_frames
    positions = np.linspace(0, frame_count - 1, num=num_frames)
    return np.clip(np.round(positions).astype(int), 0, frame_count - 1).tolist()


def clear_directory(path):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def persist_progress(extracted_records, skipped_records):
    pd.DataFrame(extracted_records).to_csv(BASE_MANIFEST_PATH, index=False)
    pd.DataFrame(skipped_records).to_csv(SKIPPED_REPORT_PATH, index=False)


def save_tensor_frames_to_jpg(frames_hwc_uint8, output_dir, image_size, jpeg_quality):
    if not isinstance(frames_hwc_uint8, torch.Tensor):
        frames_hwc_uint8 = torch.as_tensor(frames_hwc_uint8)

    if frames_hwc_uint8.ndim == 3:
        frames_hwc_uint8 = frames_hwc_uint8.unsqueeze(0)

    tensor = frames_hwc_uint8
    if tensor.device.type != DEVICE.type:
        tensor = tensor.to(DEVICE, non_blocking=True)

    tensor = tensor.permute(0, 3, 1, 2).float() / 255.0
    tensor = F.interpolate(
        tensor,
        size=(image_size, image_size),
        mode="bilinear",
        align_corners=False,
    )
    tensor = (tensor.clamp(0, 1) * 255.0).byte().permute(0, 2, 3, 1).cpu().numpy()

    saved = 0
    for output_index, frame in enumerate(tensor):
        frame_bgr = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
        frame_path = output_dir / f"{output_index:04d}.jpg"
        write_ok = cv2.imwrite(
            str(frame_path),
            frame_bgr,
            [int(cv2.IMWRITE_JPEG_QUALITY), int(jpeg_quality)],
        )
        if write_ok:
            saved += 1
    return saved


def extract_uniform_frames_cpu(video_path, output_dir, num_frames, image_size, jpeg_quality, overwrite=False):
    if overwrite:
        clear_directory(output_dir)
    else:
        output_dir.mkdir(parents=True, exist_ok=True)
        existing = sorted(output_dir.glob("*.jpg"))
        if len(existing) >= num_frames:
            return {"ok": True, "saved_frames": len(existing), "used_cache": True, "backend": "cache"}

    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        return {"ok": False, "reason": "open_failed"}

    frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    fps = float(capture.get(cv2.CAP_PROP_FPS) or 0.0)
    width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    if frame_count <= 0:
        frame_count = 1

    targets = uniform_indices(frame_count, num_frames)
    pending = {}
    for output_index, frame_index in enumerate(targets):
        pending.setdefault(int(frame_index), []).append(output_index)

    selected_frames = []
    current_index = 0
    ok, frame = capture.read()
    if not ok or frame is None:
        capture.release()
        return {"ok": False, "reason": "first_frame_failed"}

    if 0 in pending:
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        for _ in pending.pop(0):
            selected_frames.append(rgb.copy())

    last_target = max(targets) if targets else 0
    while pending and current_index < last_target:
        ok, frame = capture.read()
        current_index += 1
        if not ok or frame is None:
            break
        output_indices = pending.pop(current_index, None)
        if output_indices is not None:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            for _ in output_indices:
                selected_frames.append(rgb.copy())

    capture.release()

    if not selected_frames:
        return {"ok": False, "reason": "no_frames_saved"}

    saved = save_tensor_frames_to_jpg(
        np.stack(selected_frames, axis=0),
        output_dir=output_dir,
        image_size=image_size,
        jpeg_quality=jpeg_quality,
    )

    return {
        "ok": saved > 0,
        "reason": "ok" if saved > 0 else "no_frames_saved",
        "saved_frames": int(saved),
        "used_cache": False,
        "backend": "cpu_sequential",
        "frame_count": int(frame_count),
        "fps": fps,
        "width": width,
        "height": height,
        "duration_sec": (frame_count / fps) if fps and fps > 0 else np.nan,
    }


def try_init_decord():
    try:
        import decord
        from decord import VideoReader, gpu

        decord.bridge.set_bridge("torch")
        _ = gpu(0)
        return {
            "available": True,
            "VideoReader": VideoReader,
            "gpu": gpu,
        }
    except Exception as exc:
        return {
            "available": False,
            "error": str(exc),
        }


DECORD_STATE = try_init_decord()
print("decord gpu backend available:", DECORD_STATE["available"])
if not DECORD_STATE["available"]:
    print("decord init error:", DECORD_STATE.get("error"))


def extract_uniform_frames_gpu(video_path, output_dir, num_frames, image_size, jpeg_quality, overwrite=False):
    if overwrite:
        clear_directory(output_dir)
    else:
        output_dir.mkdir(parents=True, exist_ok=True)
        existing = sorted(output_dir.glob("*.jpg"))
        if len(existing) >= num_frames:
            return {"ok": True, "saved_frames": len(existing), "used_cache": True, "backend": "cache"}

    if not DECORD_STATE["available"]:
        return {"ok": False, "reason": "decord_gpu_unavailable"}

    try:
        VideoReader = DECORD_STATE["VideoReader"]
        gpu = DECORD_STATE["gpu"]

        reader = VideoReader(str(video_path), ctx=gpu(0))
        frame_count = len(reader)
        if frame_count <= 0:
            return {"ok": False, "reason": "empty_video"}

        fps = float(reader.get_avg_fps() or 0.0)
        first_frame = reader[0]
        if not isinstance(first_frame, torch.Tensor):
            first_frame = torch.as_tensor(first_frame.asnumpy())
        height, width = int(first_frame.shape[0]), int(first_frame.shape[1])

        target_indices = uniform_indices(frame_count, num_frames)
        batch = reader.get_batch(target_indices)
        if not isinstance(batch, torch.Tensor):
            batch = torch.as_tensor(batch.asnumpy())

        saved = save_tensor_frames_to_jpg(
            batch,
            output_dir=output_dir,
            image_size=image_size,
            jpeg_quality=jpeg_quality,
        )

        return {
            "ok": saved > 0,
            "reason": "ok" if saved > 0 else "no_frames_saved",
            "saved_frames": int(saved),
            "used_cache": False,
            "backend": "decord_gpu",
            "frame_count": int(frame_count),
            "fps": fps,
            "width": width,
            "height": height,
            "duration_sec": (frame_count / fps) if fps and fps > 0 else np.nan,
        }
    except Exception as exc:
        return {"ok": False, "reason": f"gpu_extract_failed: {exc}"}


def extract_uniform_frames(video_path, output_dir, num_frames, image_size, jpeg_quality, overwrite=False):
    gpu_result = extract_uniform_frames_gpu(
        video_path=video_path,
        output_dir=output_dir,
        num_frames=num_frames,
        image_size=image_size,
        jpeg_quality=jpeg_quality,
        overwrite=overwrite,
    )
    if gpu_result.get("ok") or gpu_result.get("used_cache"):
        return gpu_result

    cpu_result = extract_uniform_frames_cpu(
        video_path=video_path,
        output_dir=output_dir,
        num_frames=num_frames,
        image_size=image_size,
        jpeg_quality=jpeg_quality,
        overwrite=overwrite,
    )
    if not cpu_result.get("ok"):
        cpu_result["gpu_reason"] = gpu_result.get("reason")
    return cpu_result


decord gpu backend available: True


In [ ]:
candidate_df = labels_df[labels_df["has_video_file"]].copy().reset_index(drop=True)

extracted_records = []
skipped_records = []
cache_hits = 0
backend_counts = {}

progress_bar = tqdm(
    candidate_df.itertuples(index=False),
    total=len(candidate_df),
    desc="extract_frames_gpu",
)

for row_index, row in enumerate(progress_bar, start=1):
    source_video_id = str(row.source_video_id)
    video_path = Path(row.video_path)
    frame_dir = FRAMES_ROOT / source_video_id

    result = extract_uniform_frames(
        video_path=video_path,
        output_dir=frame_dir,
        num_frames=CFG["extract_frames_per_video"],
        image_size=CFG["extract_size"],
        jpeg_quality=CFG["jpeg_quality"],
        overwrite=CFG["overwrite_existing_frames"],
    )

    if result.get("used_cache"):
        cache_hits += 1

    backend_name = result.get("backend", "failed")
    backend_counts[backend_name] = backend_counts.get(backend_name, 0) + 1

    if not result.get("ok"):
        skipped_records.append(
            {
                "source_video_id": source_video_id,
                "video_path": str(video_path),
                "reason": result.get("reason", "unknown"),
                "gpu_reason": result.get("gpu_reason", ""),
            }
        )
    else:
        record = {
            "video_key": source_video_id,
            "source_video_id": source_video_id,
            "video_path": str(video_path),
            "frame_dir": str(frame_dir),
            "is_augmented": 0,
            "aug_index": 0,
            "frame_count_extracted": int(result.get("saved_frames", 0)),
            "fps": result.get("fps", np.nan),
            "duration_sec": result.get("duration_sec", np.nan),
            "width": int(result.get("width", 0) or 0),
            "height": int(result.get("height", 0) or 0),
        }
        for label in label_columns:
            record[label] = int(getattr(row, label))
        extracted_records.append(record)

    if row_index % CFG["checkpoint_every"] == 0:
        persist_progress(extracted_records, skipped_records)

    progress_bar.set_postfix(
        done=len(extracted_records),
        skipped=len(skipped_records),
        cache=cache_hits,
        backend=backend_name,
    )

persist_progress(extracted_records, skipped_records)

base_manifest_df = pd.DataFrame(extracted_records)
skipped_df = pd.DataFrame(skipped_records)

print(f"base manifest rows: {len(base_manifest_df):,}")
print(f"skipped rows: {len(skipped_df):,}")
print(f"cache hits: {cache_hits:,}")
print("backend counts:", backend_counts)
print("base manifest:", BASE_MANIFEST_PATH)
print("skipped report:", SKIPPED_REPORT_PATH)

display(base_manifest_df.head())
display(skipped_df.head())


extract_frames_gpu:   0%|          | 0/5118 [00:00<?, ?it/s]